# 06.1 — Tokenization

How text becomes the integers a language model actually sees: characters vs words vs subwords, Byte Pair Encoding built from scratch,
the tokenizers real models use (GPT, BERT, T5, Qwen), special tokens and chat templates, and why token counts differ so much between languages.

## What's in this notebook

- 1\. Why tokenization matters
- 2\. Characters and words: the two extremes
- 3\. Byte Pair Encoding from scratch
- 4\. Byte-level BPE, WordPiece and Unigram
- 5\. Real tokenizers side by side
- 6\. Special tokens, padding and chat templates
- 7\. Tokens, languages and cost
- 8\. Tokenizer quirks behind model mistakes
- 9\. Training a tokenizer with Hugging Face, and vocabulary size
- 10\. Trade-offs
- 11\. Interview questions
- 12\. In my own words

In [1]:
import hashlib
import logging
import os
import re
import time
import urllib.request
from collections import Counter
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import tiktoken  # noqa: E402
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

DATA = Path("../data/shakespeare/input.txt")
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, DATA)
assert hashlib.md5(DATA.read_bytes()).hexdigest() == "6fb458f1232090904fb40fe944165e91", "unexpected file"
text = DATA.read_text()
split = int(len(text) * 0.9)
train_text, val_text = text[:split], text[split:]
print(f"Tiny Shakespeare: {len(text):,} characters (train {len(train_text):,}, validation {len(val_text):,})")
print(text[:180])

Tiny Shakespeare: 1,115,394 characters (train 1,003,854, validation 111,540)
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First


# 1. WHY TOKENIZATION MATTERS

A neural network works on numbers, so text is first split into **tokens**, and each token gets an integer id from a fixed **vocabulary**.
The model never sees letters, only these ids (each mapped to a learned vector by an embedding layer, 06.2).

The tokenizer decides a lot:
- **sequence length**, and so the cost: attention grows with the square of the number of tokens, and APIs charge per token;
- **what the model can represent**: a word the tokenizer can't express becomes an unknown token;
- **what the model finds hard**: spelling, counting letters and arithmetic all depend on how text is split (section 8).

| Level | Vocabulary | Sequence length | Unknown words |
|---|---|---|---|
| characters | tiny (~100) | very long | none |
| words | huge (hundreds of thousands) | short | common (new words, typos, names) |
| **subwords** (BPE, WordPiece, Unigram) | medium (30k–200k) | medium | none with byte-level BPE |

Every modern language model uses subwords: frequent words stay whole, rare words are built from pieces.

# 2. CHARACTERS AND WORDS: THE TWO EXTREMES

In [2]:
chars = sorted(set(train_text))
words_train = train_text.split()
words_val = val_text.split()
vocab_words = set(words_train)
unknown = sum(w not in vocab_words for w in words_val) / len(words_val)

print(f"characters: vocabulary {len(chars)}, validation length {len(val_text):,} tokens")
print(f"words:      vocabulary {len(vocab_words):,}, validation length {len(words_val):,} tokens, "
      f"{unknown:.1%} of validation words never seen in training")

characters: vocabulary 65, validation length 111,540 tokens
words:      vocabulary 23,841, validation length 20,153 tokens, 11.7% of validation words never seen in training


Characters need only 65 symbols but turn the validation text into 111,540 tokens. Words shorten it to 20,153 tokens, but need a 23,841-word vocabulary, and 11.7% of validation words were still never seen in training (new words, names, words with punctuation attached). Subwords sit in between.

# 3. BYTE PAIR ENCODING FROM SCRATCH

**BPE** builds a vocabulary bottom-up:
1. Start with the 256 possible **bytes** as tokens, so any text, in any language, can be represented.
2. Count every pair of adjacent tokens in the training text and **merge the most frequent pair** into a new token.
3. Repeat until the vocabulary has the size you want. The ordered list of merges *is* the tokenizer.

To encode new text, apply the learned merges in the same order. To decode, concatenate the bytes.
Like GPT-2, the text is first split into words that keep their leading space (" the"), so merges never cross word boundaries.

In [3]:
PRETOKENIZE = re.compile(r" ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")      # a simpler version of GPT-2's pattern


def merge_pair(symbols: list[bytes], pair: tuple[bytes, bytes]) -> list[bytes]:
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return out


def train_bpe(text: str, num_merges: int) -> list[tuple[bytes, bytes]]:
    word_counts = Counter(PRETOKENIZE.findall(text))                          # count each distinct word once
    splits = {w: [bytes([b]) for b in w.encode("utf-8")] for w in word_counts}  # every word starts as single bytes
    merges = []
    for _ in range(num_merges):
        pair_counts = Counter()
        for word, count in word_counts.items():
            symbols = splits[word]
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += count
        if not pair_counts:
            break
        best = max(pair_counts, key=pair_counts.get)                         # the most frequent pair
        merges.append(best)
        splits = {w: merge_pair(s, best) for w, s in splits.items()}
    return merges


start = time.perf_counter()
merges = train_bpe(train_text, num_merges=300)
print(f"learned {len(merges)} merges in {time.perf_counter() - start:.1f} s; vocabulary = 256 bytes + {len(merges)} = {256 + len(merges)}")
print("first 12 merges:", [(a + b).decode() for a, b in merges[:12]])
print("merges 290-299:", [(a + b).decode() for a, b in merges[290:300]])

learned 300 merges in 9.2 s; vocabulary = 256 bytes + 300 = 556
first 12 merges: [' t', 'he', ' a', 'ou', ' s', ' m', 'in', ' w', 're', 'ha', ' the', 'nd']
merges 290-299: ['As', 'ish', ' r', 'ink', 'ig', 'ort', ' may', 'fore', ' A', 'ves']


Encoding applies the merges by priority (earliest learned first); each token then maps to an id.

In [4]:
def build_vocab(merges):
    vocab = {bytes([i]): i for i in range(256)}
    for a, b in merges:
        vocab[a + b] = len(vocab)
    return vocab


def encode(text: str, merges) -> list[bytes]:
    rank = {pair: i for i, pair in enumerate(merges)}
    tokens = []
    for word in PRETOKENIZE.findall(text):
        symbols = [bytes([b]) for b in word.encode("utf-8")]
        while len(symbols) > 1:
            best = min(zip(symbols, symbols[1:]), key=lambda p: rank.get(p, float("inf")))
            if best not in rank:
                break
            symbols = merge_pair(symbols, best)
        tokens.extend(symbols)
    return tokens


vocab = build_vocab(merges)
inverse = {i: tok for tok, i in vocab.items()}
sentence = "To be, or not to be: that is the tokenization question."
pieces = encode(sentence, merges)
ids = [vocab[p] for p in pieces]
print("pieces:", [p.decode() for p in pieces])
print("ids:   ", ids)
print("decoded matches the input:", b"".join(inverse[i] for i in ids).decode() == sentence)

for n in [0, 100, 300]:
    n_tokens = len(encode(val_text, merges[:n]))
    print(f"{n:>3} merges: validation text = {n_tokens:,} tokens, {len(val_text.encode()) / n_tokens:.2f} bytes per token")

pieces: ['To', ' be', ',', ' or', ' not', ' to', ' be', ':', ' that', ' is', ' the', ' to', 'k', 'en', 'i', 'z', 'at', 'ion', ' ', 'qu', 'est', 'ion', '.']
ids:    [395, 305, 44, 528, 321, 288, 305, 58, 322, 326, 266, 288, 107, 281, 105, 122, 304, 394, 32, 533, 376, 394, 46]
decoded matches the input: True
  0 merges: validation text = 111,540 tokens, 1.00 bytes per token
100 merges: validation text = 68,923 tokens, 1.62 bytes per token


300 merges: validation text = 56,627 tokens, 1.97 bytes per token


The first merges are the most common pieces of English: ' t', 'he', ' a', 'ou', and ' the' already at merge 11. Later merges build whole short words and endings (' may', 'fore', 'ish'). The unseen word 'tokenization' is still encoded, from smaller pieces (' to', 'k', 'en', 'i', 'z', 'at', 'ion'), and decoding gives back the exact input. 300 merges already shrink the validation text from 111,540 bytes to 56,627 tokens (1.97 bytes per token).

# 4. BYTE-LEVEL BPE, WORDPIECE AND UNIGRAM

| Algorithm | How it builds the vocabulary | How it marks word pieces | Used by |
|---|---|---|---|
| **byte-level BPE** | merge the most frequent pair, starting from bytes | the leading space is part of the token (`Ġhard` = " hard") | GPT-2/3/4, LLaMA 3, Qwen, Mistral (via tiktoken or Hugging Face `tokenizers`) |
| **WordPiece** | like BPE, but merges the pair that most increases the training data's likelihood | pieces inside a word start with `##` (`token`, `##ization`) | BERT, DistilBERT |
| **Unigram** (SentencePiece) | start with a large vocabulary and remove the tokens that matter least, by likelihood | spaces become `▁` (`▁hard`) | T5, ALBERT, XLNet, older LLaMA (SentencePiece BPE) |

**SentencePiece** is a library that works on raw text including spaces, so it needs no language-specific word splitting (useful for Chinese, Japanese and Thai). It can run BPE or Unigram.
Byte-level tokenizers never produce an unknown token; character-based vocabularies such as BERT's need an `[UNK]` token for characters they lack.

# 5. REAL TOKENIZERS SIDE BY SIDE

The same sentence through six tokenizers: GPT-2, GPT-4 (`cl100k_base`) and GPT-4o (`o200k_base`) from **tiktoken**, plus BERT, T5 and Qwen2.5 from the Hugging Face Hub.

In [5]:
tiktokens = {name: tiktoken.get_encoding(name) for name in ["gpt2", "cl100k_base", "o200k_base"]}
hf = {name: AutoTokenizer.from_pretrained(repo) for name, repo in
      [("BERT", "bert-base-uncased"), ("T5", "google-t5/t5-small"), ("Qwen2.5", "Qwen/Qwen2.5-0.5B-Instruct")]}

sample = "Tokenization isn't hard, unbelievably!"
for name, enc in tiktokens.items():
    pieces = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(sample)]
    print(f"{name:<12} vocab {enc.n_vocab:>7,}  {len(pieces):>2} tokens  {pieces}")
for name, tok in hf.items():
    pieces = tok.tokenize(sample)
    print(f"{name:<12} vocab {len(tok):>7,}  {len(pieces):>2} tokens  {pieces}")

gpt2         vocab  50,257   8 tokens  ['Token', 'ization', ' isn', "'t", ' hard', ',', ' unbelievably', '!']
cl100k_base  vocab 100,277   9 tokens  ['Token', 'ization', ' isn', "'t", ' hard', ',', ' unbelie', 'vably', '!']
o200k_base   vocab 200,019   7 tokens  ['Token', 'ization', " isn't", ' hard', ',', ' unbelievably', '!']
BERT         vocab  30,522  13 tokens  ['token', '##ization', 'isn', "'", 't', 'hard', ',', 'un', '##bel', '##ie', '##va', '##bly', '!']
T5           vocab  32,100  14 tokens  ['▁To', 'ken', 'ization', '▁is', 'n', "'", 't', '▁hard', ',', '▁unbe', 'lie', 'v', 'ably', '!']
Qwen2.5      vocab 151,665   9 tokens  ['Token', 'ization', 'Ġisn', "'t", 'Ġhard', ',', 'Ġunbelie', 'vably', '!']


All six handle the sentence, in different ways. The GPT tokenizers keep ' unbelievably' whole or in two pieces, and o200k even keeps " isn't" as one token. BERT (WordPiece, lowercased) marks pieces inside a word with `##`, and T5 (Unigram via SentencePiece) marks word starts with `▁`; their smaller vocabularies need 13–14 tokens. Qwen's output shows the raw byte-level form, where `Ġ` stands for a leading space.

# 6. SPECIAL TOKENS, PADDING AND CHAT TEMPLATES

Tokenizers add **special tokens** that the model was trained with:

| Token | Model | Meaning |
|---|---|---|
| `[CLS]`, `[SEP]` | BERT | start of input (its vector is used for classification), separator between two texts |
| `[PAD]` and an **attention mask** | most models | fill shorter sequences in a batch; the mask (1 = real, 0 = padding) tells attention to ignore them |
| `[MASK]` | BERT | the hidden word to predict during pre-training |
| `<|endoftext|>`, BOS / EOS | GPT-style models | the boundary between documents, or the beginning and end of a sequence |
| `<|im_start|>`, `<|im_end|>` and similar | chat models | the start and end of each message in a conversation |

A **chat template** turns a list of messages into the exact text format the model was fine-tuned on. Using the wrong format makes a chat model noticeably worse,
so always build prompts with `tokenizer.apply_chat_template` (or the provider's API, which does it for you).

In [6]:
bert = hf["BERT"]
batch = bert(["Hello world", "A slightly longer sentence here"], padding=True)
for ids, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(f"{str(bert.convert_ids_to_tokens(ids)):<70} mask {mask}")

gpt2 = tiktokens["gpt2"]
print("\nGPT-2 end-of-text token:", gpt2.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

qwen = hf["Qwen2.5"]
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is BPE?"}]
prompt = qwen.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"\nQwen2.5 chat template ({len(qwen.encode(prompt))} tokens):\n{prompt}")

['[CLS]', 'hello', 'world', '[SEP]', '[PAD]', '[PAD]', '[PAD]']        mask [1, 1, 1, 1, 0, 0, 0]
['[CLS]', 'a', 'slightly', 'longer', 'sentence', 'here', '[SEP]']      mask [1, 1, 1, 1, 1, 1, 1]

GPT-2 end-of-text token: [50256]

Qwen2.5 chat template (24 tokens):
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is BPE?<|im_end|>
<|im_start|>assistant



BERT adds `[CLS]` and `[SEP]` and pads the shorter sentence, with zeros in its attention mask. The chat template wraps each message in `<|im_start|>role ... <|im_end|>`
and ends with an open assistant turn, which is where the model starts writing its answer.

# 7. TOKENS, LANGUAGES AND COST

The same meaning can take very different numbers of tokens. Tokenizers trained mostly on English split other scripts into many small pieces,
which makes those languages slower, more expensive per word and shorter in effective context.

In [7]:
samples = {
    "English": "The weather is very nice today, so we are going to the market.",
    "Hindi": "आज मौसम बहुत अच्छा है, इसलिए हम बाज़ार जा रहे हैं।",
    "Marathi": "आज हवामान खूप छान आहे, म्हणून आम्ही बाजारात जात आहोत.",
    "Python code": "def add(a, b):\n    return a + b",
    "a number": "The total is 1234567.89 rupees.",
}
print(f"{'':<12}{'chars':>6}{'gpt2':>7}{'cl100k':>8}{'o200k':>7}{'Qwen2.5':>9}")
for name, s in samples.items():
    counts = [len(enc.encode(s)) for enc in tiktokens.values()] + [len(qwen.encode(s))]
    print(f"{name:<12}{len(s):>6}" + "".join(f"{n:>{w}}" for n, w in zip(counts, [7, 8, 7, 9])))

             chars   gpt2  cl100k  o200k  Qwen2.5
English         62     15      15     15       15
Hindi           50     75      54     15       50
Marathi         53     79      57     20       53
Python code     31     15      11     11       11
a number        31     11      12     12       17


- **English** costs 15 tokens in every tokenizer.
- **Hindi** costs 75 tokens with GPT-2, 54 with cl100k and 50 with Qwen2.5 (one token per character), but only 15 with o200k, the same as English. Marathi follows the same pattern (79 down to 20). With an English-centric tokenizer, a Hindi user pays about 5 times more per sentence, waits longer for each answer, and fits 5 times less text into the context window.
- **Code** is cheaper in the newer tokenizers, which merge runs of spaces.
- **Numbers:** Qwen splits them into single digits (17 tokens for the number sentence), a choice made to help arithmetic.

# 8. TOKENIZER QUIRKS BEHIND MODEL MISTAKES

Some well-known LLM failures come straight from tokenization:

In [8]:
enc = tiktokens["cl100k_base"]
show = lambda s: [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(s)]

print("letters hidden inside tokens:", show("strawberry"), "-> 'how many r's?' is hard when the model never sees the letters")
print("digits in chunks of three:   ", show("12345678 + 87654321"))
print("leading space changes the id:", enc.encode("hello"), "vs", enc.encode(" hello"))
print("case changes the tokens:     ", show("Hello"), show("HELLO"), show("hElLo"))
print("trailing space:              ", show("The answer is"), "vs", show("The answer is "))

letters hidden inside tokens: ['str', 'aw', 'berry'] -> 'how many r's?' is hard when the model never sees the letters
digits in chunks of three:    ['123', '456', '78', ' +', ' ', '876', '543', '21']
leading space changes the id: [15339] vs [24748]
case changes the tokens:      ['Hello'] ['HEL', 'LO'] ['h', 'El', 'Lo']
trailing space:               ['The', ' answer', ' is'] vs ['The', ' answer', ' is', ' ']


- **Spelling and counting letters:** "strawberry" arrives as a few multi-letter tokens, so the model never sees the individual letters.
- **Arithmetic:** numbers are cut into chunks that don't line up with place value, which makes column-by-column arithmetic awkward. Newer tokenizers split digits more consistently, and tools (a calculator or code) are the reliable fix.
- **Spaces and case:** `"hello"` and `" hello"` are different tokens, and unusual capitalization breaks words into rare pieces. A prompt ending in a trailing space can push the model into an unusual token and odd output.

# 9. TRAINING A TOKENIZER WITH HUGGING FACE, AND VOCABULARY SIZE

The `tokenizers` library (written in Rust) trains the same byte-level BPE as section 3, much faster. You train your own tokenizer when building a model
from scratch for a new language or domain; when fine-tuning an existing model, you must keep **its** tokenizer, because the embeddings are tied to its ids.

In [9]:
def train_hf_bpe(vocab_size: int) -> Tokenizer:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["<|endoftext|>"],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tok.train_from_iterator(train_text.splitlines(), trainer)
    return tok


val_bytes = len(val_text.encode())
print(f"{'tokenizer':<32}{'vocabulary':>11}{'validation tokens':>19}{'bytes per token':>17}")
ours = len(encode(val_text, merges))
print(f"{'ours, from scratch (section 3)':<32}{256 + len(merges):>11,}{ours:>19,}{val_bytes / ours:>17.2f}")
for size in [1000, 4000, 16000]:
    start = time.perf_counter()
    tok = train_hf_bpe(size)
    n = len(tok.encode(val_text).ids)
    print(f"{f'HF BPE on Shakespeare ({time.perf_counter() - start:.1f} s)':<32}{tok.get_vocab_size():>11,}{n:>19,}{val_bytes / n:>17.2f}")
n = len(tiktokens["gpt2"].encode(val_text))
print(f"{'GPT-2 (trained on web text)':<32}{tiktokens['gpt2'].n_vocab:>11,}{n:>19,}{val_bytes / n:>17.2f}")
print("round trip:", tok.decode(tok.encode(sentence).ids) == sentence)

tokenizer                        vocabulary  validation tokens  bytes per token
ours, from scratch (section 3)          556             56,627             1.97


HF BPE on Shakespeare (0.3 s)         1,000             49,671             2.25


HF BPE on Shakespeare (0.3 s)         4,000             38,542             2.89


HF BPE on Shakespeare (0.4 s)        16,000             33,720             3.31
GPT-2 (trained on web text)          50,257             36,059             3.09
round trip: True


Hugging Face's trainer builds a 16,000-token vocabulary in under half a second, against 9 seconds for our 300 merges in pure Python. A bigger vocabulary means fewer tokens: 1.97 bytes per token with our 556 tokens, 3.31 with 16,000. The 16,000-token Shakespeare tokenizer even beats GPT-2's 50,257 tokens on this text (33,720 against 36,059 tokens), because it was trained on exactly this kind of text, while GPT-2's vocabulary is spread across the whole web.

**Choosing a vocabulary size** is a trade-off. A larger vocabulary means shorter sequences (cheaper attention, more text per context window) and better coverage of many languages,
but a bigger embedding matrix and output layer (vocabulary × model width: 200,000 × 4,096 ≈ 820 million parameters), and rare tokens that get few training examples.
English-centric models used about 32k–50k tokens; recent multilingual LLMs use 128k–256k.

# 10. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Token level | characters / words / subwords | characters: no unknowns, very long sequences. Words: short, but a huge vocabulary and many unknown words. Subwords: the balance | subwords |
| Algorithm | byte-level BPE / WordPiece / Unigram | byte-level BPE: never unknown, the LLM standard. WordPiece: BERT family. Unigram: probabilistic, good for languages without spaces | byte-level BPE for new LLMs |
| Vocabulary size | small (~32k) / large (~200k) | small: smaller embeddings, longer sequences, other languages fragment. Large: shorter sequences and fairer multilingual costs, bigger embedding and softmax | 128k–256k for multilingual LLMs |
| Own tokenizer vs existing | train one / reuse the model's | training: fits a new language or domain, only possible when training a model from scratch. Reusing: required for fine-tuning a pretrained model | reuse; train only for new models |
| Prompt format | hand-written / `apply_chat_template` | hand-written: easy to get subtly wrong, which degrades chat models. Template: the exact training format | `apply_chat_template` or the provider's API |
| Counting tokens | estimate (~4 characters per token) / the real tokenizer | estimate: fine for English, badly off for other languages and code. Real tokenizer: exact | the real tokenizer for limits and costs |

# 11. INTERVIEW QUESTIONS

### Q1. What is tokenization, and why do LLMs use subwords, in 30 seconds?
Tokenization splits text into units and maps each to an integer id from a fixed vocabulary; the model only sees ids. Characters make sequences too long and words make the vocabulary huge
with many unknowns. Subwords (BPE, WordPiece, Unigram) keep frequent words whole and build rare ones from pieces, giving a manageable vocabulary, reasonable sequence lengths and no unknown words with byte-level BPE.

### Q2. How does BPE work?
Start with single bytes (or characters) as the vocabulary. Repeatedly count adjacent token pairs in the training corpus and merge the most frequent pair into a new token,
until the vocabulary reaches the target size. Encoding applies the learned merges in order; decoding concatenates the bytes.

### Q3. BPE vs WordPiece vs Unigram?
BPE merges the most frequent pair. WordPiece merges the pair that most increases the corpus likelihood and marks word-internal pieces with `##` (BERT).
Unigram starts with a large vocabulary and prunes the least useful tokens by likelihood (SentencePiece, T5). All produce subword vocabularies; byte-level BPE dominates modern LLMs.

### Q4. Why byte-level BPE?
Starting from the 256 bytes means any string in any language, including emoji and code, can be encoded, so there's never an unknown token.

### Q5. What are special tokens and the attention mask?
Reserved tokens with roles the model learned in training: `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`, BOS/EOS, `<|endoftext|>`, and chat markers.
The attention mask marks real tokens (1) versus padding (0), so attention and the loss ignore the padding.

### Q6. What is a chat template, and why does it matter?
The exact text format, with role markers and special tokens, that a chat model was fine-tuned on. Prompts built in a different format are out of distribution and get worse answers,
so use `apply_chat_template` or the provider's API.

### Q7. Why do LLMs struggle to count the letters in "strawberry" or to do arithmetic?
They see tokens, not letters: "strawberry" is a few multi-letter tokens. Numbers are split into chunks that don't match place value.
Fixes include digit-level tokenization, chain-of-thought that spells things out, or calling a tool such as a calculator or code.

### Q8. Why does the same text cost more tokens in Hindi than in English?
Tokenizers learn merges from their training data. If it's mostly English, Devanagari text gets few merges and is split into many small pieces, sometimes one token per character or even per byte.
Larger, multilingual vocabularies (o200k) bring Hindi close to English (section 7).

### Q9. How do you choose a vocabulary size?
Larger vocabularies shorten sequences and cover more languages fairly, but enlarge the embedding and output layers and leave rare tokens undertrained.
Smaller ones are cheaper per token but produce longer sequences. Modern multilingual LLMs use roughly 128k–256k.

### Q10. Can you change the tokenizer when fine-tuning a pretrained model?
Not freely: the embedding matrix is indexed by the tokenizer's ids, so a different tokenizer breaks the mapping. You can add a few new tokens (and resize the embeddings),
but they start untrained. To change tokenizers completely you'd need substantial further training.

### Q11. How would you estimate the cost of an LLM feature?
Count tokens with the model's own tokenizer on realistic inputs and outputs (prompt, retrieved context, answer), multiply by the per-token prices for input and output,
and by the expected request volume. Don't rely on "4 characters per token" for non-English text or code.

### Q12. What's the difference between `tokenize`, `encode` and `decode`?
`tokenize` returns the token strings (pieces); `encode` returns integer ids, usually with special tokens added; `decode` turns ids back into text.
For byte-level tokenizers, `decode(encode(text))` returns the original text exactly.

# 12. IN MY OWN WORDS

Write this from memory, without looking above:

- Why subwords, instead of characters or words:
- The BPE training loop, step by step:
- What special tokens, attention masks and chat templates are for:
- Why token counts differ between languages, and why it matters:
- Two model mistakes caused by tokenization: